# M4S3 - HPC and Machine Learning
## Training is not only an ML problem - it is a systems problem

**Predict → run → observe → discuss**

Core and optional examples:
1. environment and accelerator check;
2. tiny ML training loop;
3. Python loops vs optimized NumPy;
4. preprocessing bottleneck;
5. batch-size trade-off;
6. prefetch/overlap;
7. precision and memory;
8. model training-state memory budget;
9. gradient accumulation;
10. checkpointing;
11. real PyTorch training;
12. real PyTorch DistributedDataParallel with two processes;
13. TensorFlow MirroredStrategy;
14. distributed strategy map, including multi-worker training;
15. Slurm/GPU resource inspection.

> The required path remains CPU-safe. Framework-specific cells use fresh cluster module environments and fail cleanly with an explanation if the framework is not exposed correctly by the current module setup.


## 0 - Check the environment
**Predict:** CPU, memory, storage or GPU - what will limit this run?


In [ ]:
import os,sys,shutil,subprocess,time,importlib.util
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from IPython.display import display, HTML
def sh(cmd):
    p=subprocess.run(cmd,shell=True,text=True,capture_output=True); return (p.stdout+p.stderr).strip()
print("host:",sh("hostname")); print("Python:",sys.version.split()[0]); print("NumPy:",np.__version__)
print("CPU cores:",os.cpu_count()); print("Slurm:",bool(shutil.which("sbatch"))); print("nvidia-smi:",shutil.which("nvidia-smi"))
if shutil.which("nvidia-smi"): print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | head -4"))
cmd='bash -lc \'if type module >/dev/null 2>&1; then module -t avail 2>&1 | grep -Ei "pytorch|tensorflow|cuda|cudnn|nccl" | head -20; else echo "module command not visible"; fi\''
print("\nRelevant modules:\n"+sh(cmd))

def show_bars(title, items, unit="", digits=1):
    vals=[float(v) for _,v in items]
    m=max(vals) if vals else 1.0
    html=f"<div style='font-family:Arial,sans-serif;max-width:820px'><h4>{title}</h4>"
    for label,val in items:
        width=0 if m==0 else 100*float(val)/m
        value=f"{float(val):,.{digits}f}{unit}"
        html += (
            "<div style='display:grid;grid-template-columns:150px 1fr 170px;gap:10px;"
            "align-items:center;margin:5px 0'>"
            f"<div>{label}</div>"
            "<div style='background:#e8edf3;height:18px;border-radius:2px'>"
            f"<div style='width:{width:.1f}%;background:#55758f;height:18px;border-radius:2px'></div></div>"
            f"<div style='text-align:right'>{value}</div></div>"
        )
    display(HTML(html+"</div>"))

def run_module_python(module_candidates, pycode, tag):
    """Run a Python script in a fresh login shell after loading the first working module.
    Returns a result dict and prints a concise, student-friendly diagnostic.
    """
    work=Path("m4s3_outputs")
    work.mkdir(exist_ok=True)
    script=work/f"{tag}.py"
    script.write_text(pycode)
    bootstrap=r'''
if ! type module >/dev/null 2>&1; then
  for f in /etc/profile.d/lmod.sh /etc/profile.d/modules.sh /usr/share/lmod/lmod/init/bash; do
    if [ -r "$f" ]; then source "$f"; break; fi
  done
fi
'''
    attempts=[]
    for mod in module_candidates:
        shell=bootstrap+f'''
if ! type module >/dev/null 2>&1; then
  echo "MODULE_SYSTEM_NOT_AVAILABLE"
  exit 20
fi
module purge >/dev/null 2>&1 || true
if ! module load {mod}; then
  echo "MODULE_LOAD_FAILED:{mod}"
  exit 21
fi
echo "MODULE_LOADED:{mod}"
echo "PYTHON:$(command -v python)"
python "{script}"
'''
        p=subprocess.run(["bash","-lc",shell],text=True,capture_output=True)
        output=(p.stdout+p.stderr).strip()
        attempts.append((mod,p.returncode,output))
        if p.returncode==0:
            print(output)
            return {"ok":True,"module":mod,"output":output}
    print("No candidate module produced a working Python framework environment.")
    for mod,rc,output in attempts:
        tail="\n".join(output.splitlines()[-8:])
        print(f"\n[{mod}] return code {rc}\n{tail}")
    print("\nThis is an environment/module issue, not a failure of the ML concept.")
    print("Use the CPU-safe NumPy examples, or run the framework demo in the cluster environment recommended by the administrators.")
    return {"ok":False,"attempts":attempts}


## 1 - A tiny ML workload
We use a small softmax classifier. The model is intentionally simple: today we care about **data movement, batching, memory and throughput**, not network architecture.


In [ ]:
rng=np.random.default_rng(42); N,D,C=12000,96,5
X=rng.normal(size=(N,D)).astype(np.float32); wt=rng.normal(size=(D,C)).astype(np.float32)
y=np.argmax(X@wt+0.7*rng.normal(size=(N,C)),axis=1).astype(np.int64)
cut=9600; Xtr,Xv=X[:cut],X[cut:]; ytr,yv=y[:cut],y[cut:]
def softmax(z):
    z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def acc(X,y,W,b): return float(np.mean(np.argmax(X@W+b,1)==y))
def grad(X,y,W,b):
    p=softmax(X@W+b); n=len(X); loss=-np.log(p[np.arange(n),y]+1e-12).mean()
    p[np.arange(n),y]-=1; p/=n; return float(loss),X.T@p,p.sum(0)
def train(bs=128,epochs=4,lr=.18,seed=0):
    rg=np.random.default_rng(seed); W=np.zeros((D,C),np.float32); b=np.zeros(C,np.float32); updates=0; t=time.perf_counter()
    for _ in range(epochs):
        ids=rg.permutation(len(Xtr))
        for s in range(0,len(Xtr),bs):
            q=ids[s:s+bs]; _,gW,gb=grad(Xtr[q],ytr[q],W,b); W-=lr*gW; b-=lr*gb; updates+=1
    return W,b,time.perf_counter()-t,updates
W,b,t,u=train()
print(f"samples={len(Xtr):,}, features={D}, data={Xtr.nbytes/2**20:.2f} MiB")
print(f"time={t:.3f}s, updates={u}, throughput={len(Xtr)*4/t:,.0f} samples/s, val_acc={acc(Xv,yv,W,b):.3f}")


## 2 - [DEMO] Optimized libraries
**Predict:** how much faster is NumPy matrix multiplication than nested Python loops?


In [ ]:
A=rng.normal(size=(1200,256)); v=rng.normal(size=256)
def py_mv(A,v):
    out=np.empty(len(A))
    for i,row in enumerate(A):
        s=0.
        for j,x in enumerate(row): s+=x*v[j]
        out[i]=s
    return out
t=time.perf_counter(); a=py_mv(A,v); tp=time.perf_counter()-t
t=time.perf_counter(); 
for _ in range(30): b=A@v
tn=(time.perf_counter()-t)/30
print(f"Python loops: {tp:.4f}s | NumPy: {tn:.6f}s | speedup: {tp/tn:,.0f}x | same result: {np.allclose(a,b)}")


**Lesson:** PyTorch/TensorFlow Python code often calls highly optimized compiled kernels underneath.


## 3 - [DEMO] Preprocessing can starve the accelerator


In [ ]:
raw=rng.integers(0,256,size=(900,96),dtype=np.uint8)
def slow(a):
    z=np.empty(a.shape,np.float32)
    for i in range(len(a)):
        for j in range(a.shape[1]): z[i,j]=(float(a[i,j])-127.5)/127.5
    return z
def fast(a): return (a.astype(np.float32)-127.5)/127.5
t=time.perf_counter(); z1=slow(raw); ts=time.perf_counter()-t
t=time.perf_counter(); z2=fast(raw); tf=time.perf_counter()-t
print(f"slow={len(raw)/ts:,.0f} samples/s | vectorized={len(raw)/tf:,.0f} samples/s | speedup={ts/tf:.1f}x")
for prep in (30,12,4):
    compute=12; print(f"prep={prep:2} ms, compute=12 ms -> approx accelerator utilization={100*compute/(prep+compute):.1f}%")


**Fix the feeding pipeline before buying more GPUs.**


## 4 - [DEMO] Batch size: throughput ↔ memory ↔ convergence
Train the same model for the same 4 epochs.


In [ ]:
rows=[]
for bs in (16,64,256,1024):
    W,b,t,u=train(bs=bs,epochs=4,seed=1)
    rows.append((bs,u,t,len(Xtr)*4/t,acc(Xv,yv,W,b),bs*D*4/1024))
print(f"{'batch':>6} {'updates':>8} {'time':>8} {'samples/s':>12} {'val_acc':>8} {'batch_KiB':>10}")
for r in rows: print(f"{r[0]:6d} {r[1]:8d} {r[2]:8.3f} {r[3]:12,.0f} {r[4]:8.3f} {r[5]:10.1f}")


In [ ]:
show_bars(
    "Measured throughput by batch size",
    [(f"batch {r[0]}", r[3]) for r in rows],
    unit=" samples/s",
    digits=0
)
show_bars(
    "Approximate input-batch memory",
    [(f"batch {r[0]}", r[5]) for r in rows],
    unit=" KiB",
    digits=1
)
print("Validation accuracy:", {r[0]: round(r[4],3) for r in rows})
print("Question: which batch is best if throughput and validation accuracy both matter?")


Discuss:
- larger batches usually improve hardware utilization;
- memory use grows;
- updates per epoch fall;
- convergence can change;
- the batch with the highest throughput is not necessarily the batch with the best validation result.

**Batch size is both an ML parameter and a systems parameter.**


## 5 - [DEMO] Prefetch hides I/O behind compute


In [ ]:
def load(i): time.sleep(.018); return i
def compute(x): time.sleep(.030)
n=12; t=time.perf_counter()
for i in range(n): compute(load(i))
seq=time.perf_counter()-t
t=time.perf_counter()
with ThreadPoolExecutor(max_workers=1) as ex:
    f=ex.submit(load,0)
    for i in range(n):
        x=f.result()
        if i+1<n: f=ex.submit(load,i+1)
        compute(x)
pre=time.perf_counter()-t
print(f"sequential={seq:.3f}s | prefetched={pre:.3f}s | speedup={seq/pre:.2f}x")


Prefetching does not make I/O or compute individually faster; it improves **pipeline utilization** by overlapping them.


## 6 - [DEMO] Precision changes memory and numerical behaviour


In [ ]:
base=rng.normal(size=(2048,512))
for name,dtype in (("FP64",np.float64),("FP32",np.float32),("FP16",np.float16)):
    a=base.astype(dtype); print(f"{name}: {a.nbytes/2**20:.2f} MiB")
A=base[:512,:256]; B=base[:256,:256]; ref=A@B
for name,dtype in (("FP32",np.float32),("FP16",np.float16)):
    a=A.astype(dtype); b=B.astype(dtype); t=time.perf_counter(); out=a@b; dt=time.perf_counter()-t
    err=np.linalg.norm(out.astype(np.float64)-ref)/np.linalg.norm(ref)
    print(f"{name}: time={dt:.4f}s, relative_difference={err:.3e}")


In [ ]:
precision_memory=[("FP64",8.0),("FP32",4.0),("FP16",2.0)]
show_bars("Memory footprint for the same array", precision_memory, unit=" MiB", digits=1)
print("Important: lower precision reduces memory, but speed depends on hardware support.")


> On CPUs FP16 may not be faster. GPUs can have specialized tensor hardware, so **hardware support matters**.


## 7 - [DEMO] How much memory does training state need?

A model needs more memory for training than for storing its parameters alone.

For a simplified FP32 model trained with Adam:

- parameters: **4 bytes / parameter**
- gradients: **4 bytes / parameter**
- Adam first moment: **4 bytes / parameter**
- Adam second moment: **4 bytes / parameter**

Simplified training-state estimate:

**about 16 bytes per parameter**

This excludes activations, temporary buffers, framework overhead and other copies.

**Predict:** could a 7-billion-parameter model train in this simplified form on one 24 GB GPU?


In [ ]:
models=[
    ("1 million",1_000_000),
    ("100 million",100_000_000),
    ("1 billion",1_000_000_000),
    ("7 billion",7_000_000_000),
]
bytes_per_parameter=16
print(f"{'model size':>14} {'training state':>18}")
memory_rows=[]
for label,nparams in models:
    gb=nparams*bytes_per_parameter/1e9
    memory_rows.append((label,gb))
    print(f"{label:>14} {gb:15.3f} GB")
show_bars("Simplified FP32 + Adam training-state memory", memory_rows, unit=" GB", digits=1)
print("\nA 7B model needs about 112 GB for this simplified training state alone.")
print("Activations and temporary tensors would require additional memory.")


**Key idea:** model size can become an HPC memory problem before compute becomes the main problem.

Mixed precision, sharding and distributed strategies can reduce or distribute parts of this footprint. The distributed strategies belong to the next session.


## 8 - [DEMO] Gradient accumulation: a larger effective batch without a larger physical batch

Suppose GPU memory only allows a physical batch of **64**.

Instead of updating after every batch:

**64 examples → backward → update**

we can accumulate gradients:

**64 + 64 + 64 + 64 → backward four times → one update**

Effective batch size:

**64 × 4 = 256**

This helps with memory limits, but it does **not** provide the same compute speedup as four GPUs working in parallel.


In [ ]:
physical_batch=64
accumulation_steps=4
effective_batch=physical_batch*accumulation_steps
print("physical batch:", physical_batch)
print("accumulation steps:", accumulation_steps)
print("effective batch:", effective_batch)
print("Key point: memory is similar to batch 64, but the optimizer updates every 256 examples.")


**Ask:** Is gradient accumulation distributed training?

**Answer:** No. It changes when we update the model and helps with memory/effective batch size. It does not create extra accelerators or remove the serial work on one device.


## 9 - [DEMO] Checkpointing is resilience + I/O


In [ ]:
out=Path("m4s3_outputs"); out.mkdir(exist_ok=True); f=out/"checkpoint.npz"
W,b,_,_=train(bs=128,epochs=2,seed=3); before=acc(Xv,yv,W,b)
t=time.perf_counter(); np.savez(f,W=W,b=b,epoch=np.array([2])); save=time.perf_counter()-t
del W,b; t=time.perf_counter(); s=np.load(f); load=time.perf_counter()-t
after=acc(Xv,yv,s["W"],s["b"])
print(f"file={f} | size={f.stat().st_size/1024:.1f} KiB | save={save*1e3:.2f} ms | load={load*1e3:.2f} ms")
print(f"accuracy before={before:.4f}, after reload={after:.4f}, recovered={np.isclose(before,after)}")


At scale, checkpoint frequency trades **recovery time** against **storage/I/O overhead**.

Real training jobs may run for hours or days, so losing all progress after a failure can be very expensive.


## 10 - [REAL FRAMEWORK] Train a small model with PyTorch

This uses the cluster module system in a **fresh shell**, so it does not depend on the Jupyter kernel having PyTorch installed.

We measure:

- framework version;
- device actually visible to the process;
- epoch time;
- examples per second;
- training loss.

If a GPU is allocated, the same code uses CUDA. Otherwise it runs on CPU.

**Important HPC lesson:** software being installed on the cluster is different from the hardware resources allocated to your job.


In [ ]:
pytorch_train_code=r'''
import time
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

print("PyTorch:", torch.__version__)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type=="cuda":
    print("GPU:", torch.cuda.get_device_name(0))

torch.manual_seed(7)
N,D,C=12000,96,5
X=torch.randn(N,D)
true_w=torch.randn(D,C)
y=(X@true_w).argmax(dim=1)

loader=DataLoader(TensorDataset(X,y),batch_size=256,shuffle=True,num_workers=0)
model=nn.Sequential(nn.Linear(D,128),nn.ReLU(),nn.Linear(128,C)).to(device)
opt=torch.optim.SGD(model.parameters(),lr=0.08)
loss_fn=nn.CrossEntropyLoss()

for epoch in range(2):
    t0=time.perf_counter()
    seen=0
    total_loss=0.0
    for xb,yb in loader:
        xb=xb.to(device); yb=yb.to(device)
        opt.zero_grad(set_to_none=True)
        loss=loss_fn(model(xb),yb)
        loss.backward()
        opt.step()
        seen += xb.shape[0]
        total_loss += float(loss)*xb.shape[0]
    if device.type=="cuda":
        torch.cuda.synchronize()
    dt=time.perf_counter()-t0
    print(f"epoch={epoch+1} time={dt:.3f}s throughput={seen/dt:,.0f} examples/s loss={total_loss/seen:.4f}")
'''
pytorch_result=run_module_python(
    ["pytorch/2.8.0","PyTorch/2.1.2-foss-2023a"],
    pytorch_train_code,
    "pytorch_train_demo"
)


If the module runner cannot import PyTorch, the notebook now reports **which module load failed and which Python executable was used**, rather than exposing a confusing raw traceback.

For class discussion:

**Would adding a GPU help if the DataLoader or storage cannot feed batches fast enough?**


## 11 - [PREVIEW] Data-parallel gradient averaging

Session 19 will study All-Reduce and distributed training in detail.


In [ ]:
ids=rng.choice(len(Xtr),512,replace=False); xb,yb=Xtr[ids],ytr[ids]
W=np.zeros((D,C),np.float32); b=np.zeros(C,np.float32)
_,gf,bf=grad(xb,yb,W,b); gs=[]; bs=[]
for q in np.array_split(np.arange(512),4):
    _,g,bb=grad(xb[q],yb[q],W,b); gs.append(g); bs.append(bb)
ga=np.mean(gs,0); ba=np.mean(bs,0)
print("4 workers x 128 samples")
print("max gradient difference:",float(np.max(np.abs(gf-ga))))
print("max bias-gradient difference:",float(np.max(np.abs(bf-ba))))


**That averaging step is where collective communication such as All-Reduce enters.**

The numerical differences above should be extremely small because averaging the gradients from equal-sized shards reproduces the full-batch gradient up to floating-point round-off.

Session 19 will study how this synchronization is implemented and how communication limits scaling.


## 12 - [REAL DISTRIBUTED ML] PyTorch DistributedDataParallel with two processes

The previous example averaged gradients manually.

Now we use real **PyTorch DistributedDataParallel (DDP)**.

Classroom-safe version:

**2 local processes on CPU + Gloo backend**

Each process:

1. owns a full copy of the model;
2. receives a different data shard;
3. computes its local gradients;
4. DDP synchronizes gradients across processes;
5. both replicas apply the same update.

On a GPU node, the same pattern normally becomes:

**one process per GPU + NCCL backend**

For four GPUs:

`torchrun --nproc_per_node=4 train_ddp.py`


In [ ]:
pytorch_ddp_code=r'''
import os, socket
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch import nn
from torch.nn.parallel import DistributedDataParallel as DDP

def free_port():
    s=socket.socket(); s.bind(("",0)); p=s.getsockname()[1]; s.close(); return p

def worker(rank, world_size, port):
    os.environ["MASTER_ADDR"]="127.0.0.1"
    os.environ["MASTER_PORT"]=str(port)
    dist.init_process_group("gloo",rank=rank,world_size=world_size)

    torch.manual_seed(123)
    model=nn.Linear(8,2)
    ddp=DDP(model)

    # Different shard on each rank
    torch.manual_seed(1000+rank)
    x=torch.randn(32,8)
    y=torch.randint(0,2,(32,))
    loss=nn.CrossEntropyLoss()(ddp(x),y)
    loss.backward()

    # DDP has already synchronized gradients at this point.
    opt=torch.optim.SGD(ddp.parameters(),lr=0.1)
    opt.step()

    checksum=sum(p.detach().sum() for p in ddp.parameters())
    gathered=[torch.zeros_like(checksum) for _ in range(world_size)]
    dist.all_gather(gathered,checksum)
    if rank==0:
        values=[float(v) for v in gathered]
        print("world_size:", world_size)
        print("parameter checksums after synchronized update:", values)
        print("replicas match:", max(values)-min(values) < 1e-6)
    dist.destroy_process_group()

if __name__=="__main__":
    port=free_port()
    mp.spawn(worker,args=(2,port),nprocs=2,join=True)
'''
ddp_result=run_module_python(
    ["pytorch/2.8.0","PyTorch/2.1.2-foss-2023a"],
    pytorch_ddp_code,
    "pytorch_ddp_demo"
)


**What changed compared with one process?**

We gained parallel workers, but we also introduced:

- synchronization;
- collective communication;
- extra process management;
- possible load imbalance.

With multiple GPUs, DDP normally keeps **one model replica per GPU** and synchronizes gradients after backward propagation.


## 13 - [REAL FRAMEWORK] TensorFlow MirroredStrategy

**MirroredStrategy** is synchronous data parallelism within one machine.

With four visible GPUs:

- TensorFlow creates one model replica on each GPU;
- the global batch is split across replicas;
- each GPU computes forward and backward passes on its local batch;
- gradients are reduced across replicas;
- every replica receives the same updated weights.

If this Jupyter session has no GPUs, the demo attempts to create **two logical CPU devices** so students can still see the replica model and distributed API.


In [ ]:
tensorflow_mirrored_code=r'''
import time
import tensorflow as tf

print("TensorFlow:", tf.__version__)
gpus=tf.config.list_physical_devices("GPU")
print("physical GPUs:", [d.name for d in gpus])

if not gpus:
    cpus=tf.config.list_physical_devices("CPU")
    if cpus:
        try:
            tf.config.set_logical_device_configuration(
                cpus[0],
                [tf.config.LogicalDeviceConfiguration(), tf.config.LogicalDeviceConfiguration()]
            )
        except Exception as e:
            print("logical CPU configuration note:", type(e).__name__, str(e)[:160])

logical_gpus=tf.config.list_logical_devices("GPU")
logical_cpus=tf.config.list_logical_devices("CPU")
devices=[d.name for d in logical_gpus] if logical_gpus else [d.name for d in logical_cpus[:2]]
print("replica devices:", devices)

strategy=tf.distribute.MirroredStrategy(devices=devices if len(devices)>=2 else None)
print("num_replicas_in_sync:", strategy.num_replicas_in_sync)

N,D,C=4096,32,4
x=tf.random.normal((N,D),seed=7)
w=tf.random.normal((D,C),seed=8)
y=tf.argmax(x@w,axis=1)

global_batch=64*strategy.num_replicas_in_sync
ds=tf.data.Dataset.from_tensor_slices((x,y)).shuffle(1024,seed=7).batch(global_batch)

with strategy.scope():
    model=tf.keras.Sequential([
        tf.keras.layers.Input(shape=(D,)),
        tf.keras.layers.Dense(64,activation="relu"),
        tf.keras.layers.Dense(C)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.SGD(0.05),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    )

t0=time.perf_counter()
hist=model.fit(ds,epochs=2,verbose=0)
dt=time.perf_counter()-t0
print("global_batch:", global_batch)
print("final_loss:", float(hist.history["loss"][-1]))
print("train_time:", round(dt,3),"s")
'''
tensorflow_result=run_module_python(
    ["TensorFlow/2.13.0-foss-2023a","tensorflow/2.13.0-foss-2023a"],
    tensorflow_mirrored_code,
    "tensorflow_mirrored_demo"
)


**PyTorch DDP and TensorFlow MirroredStrategy implement the same basic HPC pattern:**

**replicate model → split data → compute local gradients → synchronize → update**

The API is different. The systems idea is the same.


## 14 - Which distributed strategy fits which problem?

### A. Single-device training
Use when the model and useful batch fit comfortably on one accelerator.

### B. Synchronous data parallelism
**PyTorch:** DistributedDataParallel  
**TensorFlow:** MirroredStrategy

Use when:

- the model fits on each GPU;
- the dataset/workload is large;
- you want more throughput.

### C. Multi-worker data parallelism
**PyTorch:** DDP across nodes  
**TensorFlow:** MultiWorkerMirroredStrategy

Each worker can manage one or more GPUs. Gradients must be synchronized across the network.

### D. Model/state sharding
Examples include PyTorch FSDP and other sharding approaches.

Use when a full training state is too large for one GPU.

### E. Model or pipeline parallelism
Split different parts/stages of a large model across devices.

Use when the model itself does not fit on one accelerator.

**Today:** understand the map.  
**Next session:** measure scaling, All-Reduce and communication cost.


In [ ]:
strategy_cases=[
    ("Model fits, more data/throughput needed","DDP / MirroredStrategy"),
    ("Same idea across several nodes","DDP multi-node / MultiWorkerMirroredStrategy"),
    ("Training state too large for one GPU","FSDP / sharding"),
    ("Model layers too large for one GPU","model or pipeline parallelism")
]
for problem,strategy in strategy_cases:
    print(f"{problem:44s} -> {strategy}")


### What happens with 4 GPUs using data parallelism?

**Global batch 512**

→ GPU 0 receives 128 examples  
→ GPU 1 receives 128 examples  
→ GPU 2 receives 128 examples  
→ GPU 3 receives 128 examples

All four GPUs hold a model replica and compute gradients concurrently.

Then:

**local gradients → collective reduction → synchronized gradients → same model update**

The extra GPUs reduce compute time only if the added communication and input-pipeline costs do not dominate.


## 15 - [OPTIONAL CLUSTER] What would an ML job request from Slurm?

A GPU training job usually needs more than a GPU.

Typical resources include:

- **GPU** for model computation;
- **CPU cores** for loading and preprocessing;
- **memory** for batches and worker processes;
- **storage bandwidth** for datasets and checkpoints.

A generic request might look like:

```bash
#SBATCH --job-name=ml-demo
#SBATCH --gpus=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
```

The exact GPU partition and options are cluster-specific, so inspect the system before submitting.


In [ ]:
if shutil.which("sinfo"):
    print("Visible Slurm partitions and generic resources:")
    print(sh("sinfo -o '%P  GRES=%G  CPUs=%c  MemMB=%m' | head -15"))
else:
    print("sinfo is not visible in this environment.")

print("\nQuestion: why request CPU cores for a GPU training job?")
print("Expected: loading, decoding, preprocessing, augmentation and orchestration can run on CPUs.")


# 16 - Exit ticket

A training job shows:
- GPU utilization **35%**
- GPU memory **45%**
- CPU utilization **100%**
- continuous storage reads

**Would adding more GPUs be your first change? Why or why not?**

Expected: **probably not**. The job appears CPU/data-feed limited, so additional GPUs may simply wait.

### Remember

**pipeline → bottleneck → evidence → change one factor → remeasure**

### Bridge to M4S4

One GPU can run out of compute capacity or memory. Multiple GPUs can help, but they introduce **communication and synchronization**.

Next: **distributed ML and AI workflows**
